# SPY v2 — 5-session direction-only AI (fully trainable CNN)

**Goal:** Use the most recent **42 completed SPY daily candlesticks**, EMA 8, EMA 21, and volume **as image pixels only**. Predict whether the **closing price five trading sessions later** is above (**UP**) or below (**DOWN**) today's close. This is **NOT** the next candle's color or an options trade recommendation.

**Model:** A compact convolutional neural network trained **from scratch**, with **100% of weights trainable**. No pretrained ImageNet weights, no candle-body regression.

**Data policy:** Training 2020–2024, development/validation 2025, **locked 2026 final test is not loaded or evaluated**. Signals during bearish rallies near EMA 8/21 are reported separately. Adjacent 42-day charts and five-session targets overlap, so validation samples are not statistically independent.

**Checkpoints:** Each epoch saves a resumable checkpoint. The notebook uses an already mounted Drive when available. Otherwise it writes to the runtime and builds `/content/SPY-EMA-V2-checkpoints.zip` for download from the Files panel. Download the archive before the runtime expires. Set `SAVE_CHECKPOINTS_TO_DRIVE` to `True` to request a Drive mount.

**First-time use:** In Colab choose **Runtime → Change runtime type → T4 GPU** (if available), then **Runtime → Run all**. The notebook prints the training progress, GPU type, parameters and 2025 metrics. Training stops early if balanced accuracy fails to improve.

Source repo: [jamelof23/SPY-EMA-V2](https://github.com/jamelof23/SPY-EMA-V2)  ·  Image source: [original archived charts](https://github.com/jamelof23/SPY/releases/tag/spy42-data-2026-11-22-v1).


In [ ]:
# 1. Runtime and experiment configuration
from pathlib import Path
import sys, subprocess, json, os, importlib.util

REPO = 'jamelof23/SPY-EMA-V2'
REPO_DIR = Path('/content/SPY-EMA-V2')
DATA_DIR = Path('/content/spy42_v2_data')
SAVE_CHECKPOINTS_TO_DRIVE = False   # Set True to request a new Drive mount
DRIVE_DIR = Path('/content/drive/MyDrive/SPY-EMA-V2/model_seed42')
LOCAL_OUTPUT_DIR = Path('/content/spy42_v2_run')
EPOCHS = 30
BATCH_SIZE = 16
RANDOM_SEED = 42

import torch
print('PyTorch:',torch.__version__)
print('CUDA GPU available:',torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:',torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU not enabled. In Colab select Runtime > Change runtime type > T4 GPU, then rerun.')


In [ ]:
# 2. Read the audited v2 code and label CSVs from GitHub (no token needed for public repository)
if not (REPO_DIR / 'src' / 'train_v2.py').exists():
    subprocess.run(['git','clone','--depth','1',f'https://github.com/{REPO}.git',str(REPO_DIR)],check=True)
else:
    print('Repository already cloned. Existing files reused; restart the runtime to fetch updates.')

required = [REPO_DIR/'src'/'prepare_data.py',REPO_DIR/'src'/'train_v2.py',
            REPO_DIR/'data'/'labels'/'train_labels.csv',
            REPO_DIR/'data'/'labels'/'validation_labels.csv']
for p in required:
    assert p.is_file(), f'Missing: {p}'
print('Verified training script and labels in',REPO_DIR)


In [ ]:
# 3. Fetch only the published train/validation ZIP and verify it before use.
import hashlib
import urllib.request

asset_name = 'SPY42_v2_trainval_images.zip'
asset_url = f'https://github.com/{REPO}/releases/download/spy42-v2-images-v1/{asset_name}'
expected_sha = '030d0d8bc9306a80b57be0b66f9f90a0dc41c716514cc1b29964db800158095f'
DATA_DIR.mkdir(parents=True, exist_ok=True)
archive = DATA_DIR / asset_name

def archive_digest(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

if not archive.exists() or archive_digest(archive) != expected_sha:
    partial = archive.with_suffix('.part')
    try:
        urllib.request.urlretrieve(asset_url, partial)
        if archive_digest(partial) != expected_sha:
            raise ValueError('The v2 train/validation archive failed its SHA-256 check.')
        partial.replace(archive)
    finally:
        partial.unlink(missing_ok=True)

subprocess.run([sys.executable, '-u', str(REPO_DIR/'src'/'prepare_data.py'),
                '--repo-root', str(REPO_DIR), '--dest', str(DATA_DIR),
                '--archive', str(archive)], check=True)
print('Verified and prepared the 1,498 training and validation images.')


In [ ]:
# 4. Inspect class balance and one example chart (not future labeled data)
import pandas as pd
from IPython.display import display
from PIL import Image
train_rows=pd.read_csv(REPO_DIR/'data'/'labels'/'train_labels.csv')
val_rows=pd.read_csv(REPO_DIR/'data'/'labels'/'validation_labels.csv')
assert len(train_rows)==1253 and len(val_rows)==245
assert train_rows['asof_date'].str[:4].astype(int).max()<=2024
assert val_rows['asof_date'].str[:4].astype(int).eq(2025).all()
display(pd.DataFrame([
    {'split':'2020–2024 train','images':len(train_rows),'UP':int(train_rows.label_up_5d.sum()),
     'DOWN':int((1-train_rows.label_up_5d).sum())},
    {'split':'2025 validation','images':len(val_rows),'UP':int(val_rows.label_up_5d.sum()),
     'DOWN':int((1-val_rows.label_up_5d).sum())},
]))
example=DATA_DIR/'images'/'train'/'SPY_42_asof_2022-05-04.png'
with Image.open(example) as image:
    display(image.resize((768,336)))
print('Example uses past charts only. Five-session future price is used only as a 0/1 label.')


In [ ]:
# 5. Choose storage for checkpoints
if SAVE_CHECKPOINTS_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
OUTPUT_DIR = DRIVE_DIR if Path('/content/drive/MyDrive').is_dir() else LOCAL_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Checkpoint directory:', OUTPUT_DIR)


In [ ]:
# 6. Run a *short* GPU speed and network wiring benchmark (NOT full training)
subprocess.run([sys.executable,'-u',str(REPO_DIR/'src'/'train_v2.py'),
                '--repo-root',str(REPO_DIR),'--data-root',str(DATA_DIR),
                '--output-dir',str(OUTPUT_DIR),'--device','cuda',
                '--batch-size',str(BATCH_SIZE),'--seed',str(RANDOM_SEED),
                '--benchmark-only','--benchmark-batches','12'],check=True)


In [ ]:
# 7. TRAIN V2. All CNN weights learn from chart pixels. No body size task.
# Best model and resume state are saved after each completed epoch to OUTPUT_DIR.
cmd=[sys.executable,'-u',str(REPO_DIR/'src'/'train_v2.py'),
     '--repo-root',str(REPO_DIR),'--data-root',str(DATA_DIR),
     '--output-dir',str(OUTPUT_DIR),'--device','cuda',
     '--epochs',str(EPOCHS),'--batch-size',str(BATCH_SIZE),
     '--workers','2','--seed',str(RANDOM_SEED),
     '--patience','7','--lr','0.0005']
print('Starting SPY v2 GPU training. Checkpoint directory:',OUTPUT_DIR,flush=True)
from shutil import copy2
from zipfile import ZipFile, ZIP_STORED
import re

bundle = Path('/content/SPY-EMA-V2-checkpoints.zip')
def write_bundle():
    temp = bundle.with_suffix('.partial')
    names = ['best_direction_model.pt', 'last_checkpoint.pt', 'history.json',
             'training_config.json', 'validation_2025_metrics.json', 'validation_2025_predictions.csv']
    with ZipFile(temp, 'w', compression=ZIP_STORED) as archive_out:
        for name in names:
            path = OUTPUT_DIR / name
            if path.is_file():
                archive_out.write(path, name)
        for path in sorted(OUTPUT_DIR.glob('epoch_*.pt')):
            archive_out.write(path, path.name)
    temp.replace(bundle)

with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as training:
    for line in training.stdout:
        print(line, end='', flush=True)
        match = re.match(r'Epoch (\d+):', line)
        if match:
            epoch = int(match[1])
            copy2(OUTPUT_DIR/'last_checkpoint.pt', OUTPUT_DIR/f'epoch_{epoch:02d}.pt')
            if epoch == 1:
                write_bundle()
    if training.wait():
        raise RuntimeError(f'Training exited with code {training.returncode}. See output above.')
write_bundle()
print('Download from the Colab Files panel:', bundle, flush=True)


In [ ]:
# 8. Full 2025 validation report — this does NOT evaluate the reserved 2026 test
from IPython.display import Markdown,display
import matplotlib.pyplot as plt
results=json.loads((OUTPUT_DIR/'validation_2025_metrics.json').read_text())
conf=results['confusion_matrix_rows_actual_DOWN_UP_columns_predicted_DOWN_UP']
display(pd.DataFrame([
    {'metric':'Model accuracy','value':results['accuracy']},
    {'metric':'Model balanced accuracy','value':results['balanced_accuracy']},
    {'metric':'Always-UP accuracy baseline','value':results['always_up_baseline']['accuracy']},
    {'metric':'EMA8 > EMA21 baseline accuracy','value':results['subgroups_2025_asof_only']['ema8_above_ema21_direction_baseline']['accuracy']},
    {'metric':'DOWN recall','value':results['DOWN']['recall']},
    {'metric':'UP recall','value':results['UP']['recall']},
    {'metric':'Brier score','value':results['brier_score']},
]))
fig,ax=plt.subplots(figsize=(5,4))
ax.imshow(conf,cmap='Blues')
for i in range(2):
    for j in range(2): ax.text(j,i,str(conf[i][j]),ha='center',va='center',fontsize=15)
ax.set(xticks=[0,1],yticks=[0,1],xticklabels=['DOWN','UP'],yticklabels=['DOWN','UP'],
       xlabel='Predicted',ylabel='Actual',title='2025 development confusion matrix')
plt.show()
print('2025 bear-pullback context: ',results['subgroups_2025_asof_only']['bear_pullback_asof']['n'],'examples.')
print('2026 final test: NOT LOADED / NOT EVALUATED')
print('BEST EPOCH:',results['checkpoints']['best_epoch'])
print('Full metrics:',OUTPUT_DIR/'validation_2025_metrics.json')


### Important interpretation

Do not call this a profitable trading strategy based on development accuracy. The 2025 set contains correlated examples and was used to choose the model checkpoint; the 2026 outcome labels are deliberately held out. The five-session UP/DOWN target does **not** directly identify an intraday EMA rejection candle or options execution point.

For a new runtime, restore the downloaded ZIP into `LOCAL_OUTPUT_DIR` before running the training cell, or mount the Drive containing your checkpoint. The trainer resumes a matching configuration from `last_checkpoint.pt`.
